In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
import pandas as pd
import numpy as np

price_df = pd.read_csv('/content/drive/MyDrive/MRP/price_preprocessed.csv', parse_dates=['date'])
price_df = price_df.sort_values(['symbol', 'date']).reset_index(drop=True)
price_df.head(), price_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11083232 entries, 0 to 11083231
Data columns (total 8 columns):
 #   Column     Dtype         
---  ------     -----         
 0   date       datetime64[ns]
 1   volume     float64       
 2   open       float64       
 3   high       float64       
 4   low        float64       
 5   close      float64       
 6   adj close  float64       
 7   symbol     object        
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 676.5+ MB


(        date    volume      open      high       low     close  adj close  \
 0 2016-01-04  0.002927  0.004108  0.002234  0.004037  0.002770   0.002661   
 1 2016-01-05  0.002304  0.004075  0.002221  0.004037  0.002761   0.002652   
 2 2016-01-06  0.001873  0.004026  0.002223  0.004008  0.002773   0.002664   
 3 2016-01-07  0.003120  0.004016  0.002178  0.003883  0.002655   0.002551   
 4 2016-01-08  0.003327  0.003924  0.002154  0.003849  0.002627   0.002524   
 
   symbol  
 0      A  
 1      A  
 2      A  
 3      A  
 4      A  ,
 None)

In [4]:
# 1-Day Simple Return
price_df['return_1d'] = price_df.groupby('symbol')['close'].pct_change()

In [5]:
# 10-Day Moving Average of Close
price_df['ma_10'] = price_df.groupby('symbol')['close'] \
                           .transform(lambda x: x.rolling(10, min_periods=10).mean())


In [6]:
# 30-Day Rolling Volatility (std of 1-Day Returns)
price_df['vol_30'] = price_df.groupby('symbol')['return_1d'] \
                            .transform(lambda x: x.rolling(30, min_periods=30).std())

price_df[['symbol','date','return_1d','ma_10','vol_30']].head(15)


,symbol,date,return_1d,ma_10,vol_30
0,A,2016-01-04,NaN,NaN,NaN
1,A,2016-01-05,-0.003441,NaN,NaN
2,A,2016-01-06,0.004439,NaN,NaN
3,A,2016-01-07,-0.042475,NaN,NaN
4,A,2016-01-08,-0.010513,NaN,NaN
5,A,2016-01-11,-0.016844,NaN,NaN
6,A,2016-01-12,0.006589,NaN,NaN
7,A,2016-01-13,-0.034826,NaN,NaN
8,A,2016-01-14,0.020347,NaN,NaN
9,A,2016-01-15,-0.013294,0.002637,NaN


In [7]:
# 14-Day RSI
def compute_rsi(close_series, window=14):
    delta = close_series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    ma_up = up.rolling(window, min_periods=window).mean()
    ma_down = down.rolling(window, min_periods=window).mean()
    rs = ma_up / ma_down
    return 100 - (100 / (1 + rs))

price_df['rsi_14'] = price_df.groupby('symbol')['close'] \
                            .apply(lambda x: compute_rsi(x, 14)) \
                            .reset_index(level=0, drop=True)

price_df[['symbol','date','rsi_14']].head(20)

,symbol,date,rsi_14
0,A,2016-01-04,NaN
1,A,2016-01-05,NaN
2,A,2016-01-06,NaN
3,A,2016-01-07,NaN
4,A,2016-01-08,NaN
5,A,2016-01-11,NaN
6,A,2016-01-12,NaN
7,A,2016-01-13,NaN
8,A,2016-01-14,NaN
9,A,2016-01-15,NaN


In [8]:
# Drop Incomplete Windows & Save
# Remove rows where any feature is NaN (first 30 days per symbol)
price_feats = price_df.dropna(subset=['return_1d','ma_10','vol_30','rsi_14']).reset_index(drop=True)

In [10]:
# Save the enriched dataset
price_feats.to_csv('/content/drive/MyDrive/MRP/price_features_engineered.csv', index=False)

In [11]:
# Confirm
price_feats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10842070 entries, 0 to 10842069
Data columns (total 12 columns):
 #   Column     Dtype         
---  ------     -----         
 0   date       datetime64[ns]
 1   volume     float64       
 2   open       float64       
 3   high       float64       
 4   low        float64       
 5   close      float64       
 6   adj close  float64       
 7   symbol     object        
 8   return_1d  float64       
 9   ma_10      float64       
 10  vol_30     float64       
 11  rsi_14     float64       
dtypes: datetime64[ns](1), float64(10), object(1)
memory usage: 992.6+ MB
